# Regressão polinomial

**Objetivo:** ver o compromisso viés–variância ao vivo. Ajustamos polinômios de graus crescentes a dados com curvatura conhecida e acompanhamos o erro de treino e o de teste — um cai sempre, o outro faz um U.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

## 1. Dados com curvatura

Geramos pontos de uma função verdadeira suave (uma senoide) mais ruído, e separamos treino e teste. Só o treino é usado para ajustar; o teste mede generalização.

In [ ]:
from sklearn.model_selection import train_test_split

x = np.linspace(0, 1, 60)
y_verdadeiro = np.sin(2 * np.pi * x)                 # sinal real
y = y_verdadeiro + np.random.normal(0, 0.25, size=x.shape)  # + ruido

x_treino, x_teste, y_treino, y_teste = train_test_split(
    x, y, test_size=0.4, random_state=SEMENTE)
print("treino:", len(x_treino), "| teste:", len(x_teste))

## 2. Ajustar vários graus

Para cada grau, um `Pipeline` cria as potências de $x$ com `PolynomialFeatures` e ajusta uma regressão linear sobre elas. Guardamos o erro de treino e de teste de cada grau, num laço à mostra.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

graus = [1, 3, 5, 9, 15]
erro_treino = []
erro_teste = []
modelos = {}
for d in graus:
    modelo = make_pipeline(PolynomialFeatures(d), LinearRegression())
    modelo.fit(x_treino.reshape(-1, 1), y_treino)
    mse_tr = mean_squared_error(y_treino, modelo.predict(x_treino.reshape(-1, 1)))
    mse_te = mean_squared_error(y_teste, modelo.predict(x_teste.reshape(-1, 1)))
    erro_treino.append(mse_tr)
    erro_teste.append(mse_te)
    modelos[d] = modelo
    print("grau", str(d).rjust(2), "| MSE treino", round(mse_tr, 3), "| MSE teste", round(mse_te, 3))

## 3. As curvas ajustadas

Grau baixo é rígido (subajuste); grau alto serpenteia atrás do ruído (sobreajuste). A curva verdadeira está em preto.

In [ ]:
grade = np.linspace(0, 1, 200).reshape(-1, 1)
figura = go.Figure()
figura.add_trace(go.Scatter(x=x_treino, y=y_treino, mode="markers",
                            marker=dict(color=SUAVE, size=6), name="treino"))
figura.add_trace(go.Scatter(x=grade.ravel(), y=np.sin(2*np.pi*grade.ravel()),
                            mode="lines", line=dict(color=TINTA, dash="dash"), name="verdade"))
for d in [1, 5, 15]:
    figura.add_trace(go.Scatter(x=grade.ravel(), y=modelos[d].predict(grade),
                                mode="lines", name=f"grau {d}"))
figura.update_layout(title="Ajustes de diferentes graus", yaxis_range=[-1.8, 1.8],
                     height=400, margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## 4. Erro de treino × erro de teste

O gráfico que resume tudo: o erro de treino desce sempre, o de teste desenha um U. O fundo do U marca o grau que melhor generaliza.

In [ ]:
melhor = graus[int(np.argmin(erro_teste))]
figura = go.Figure()
figura.add_trace(go.Scatter(x=graus, y=erro_treino, mode="lines+markers",
                            line=dict(color=AZUL), name="erro de treino"))
figura.add_trace(go.Scatter(x=graus, y=erro_teste, mode="lines+markers",
                            line=dict(color=VERMELHO), name="erro de teste"))
figura.add_vline(x=melhor, line_dash="dash", line_color=VERDE,
                 annotation_text=f"melhor grau = {melhor}")
figura.update_layout(title="Selecao do grau: vies-variancia",
                     xaxis_title="grau do polinomio", yaxis_title="MSE",
                     height=380, margin=dict(l=10, r=10, t=50, b=10))
figura.show()
print("grau que minimiza o erro de teste:", melhor)

## Exercício

Aumente o ruído (troque `0.25` por `0.6` na geração de `y`) e rode de novo. O grau ótimo tende a subir ou a descer com mais ruído? Por quê?

<details><summary>Ver resposta</summary>

Com mais ruído, o grau ótimo tende a **descer**: como há menos sinal confiável, um modelo mais simples (mais rígido) generaliza melhor, enquanto um grau alto passa a ajustar o ruído maior e piora no teste. Mais ruído empurra a escolha para modelos menos flexíveis.

</details>